# Go2 body budget — GPU edition (Kaggle / Colab)

Thousands of simulated **Unitree Go2** robots train in parallel on one GPU. Their 12 motors heat up with
effort, lose strength when hot (derating) and cut out at the limit; the robots feel their motor
temperatures and effort, with a short memory.

* **Part 0** — speed test (5 min)
* **Part 1** — standard robot vs self-regulating robot, on 3-minute missions carrying 0 / 3 / 6 kg
* **Part 2** — evolution: does selection give robots an instinct to protect their bodies?

Project: https://github.com/pablopedrosap/homeostatic-Alife  ·  Requirements: **GPU on** and **Internet on**
(Kaggle: right panel → *Session options*).

In [ ]:
# 1) Setup — exact versions tested together (do not change them)
import os
os.environ["MUJOCO_GL"] = "egl"                       # headless rendering on the GPU server
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false" # leave GPU memory for the video renderer
!pip install -q "jax[cuda12]==0.9.2" "brax==0.14.2" "mujoco==4.0.0" "mujoco-mjx==4.0.0" mediapy

In [ ]:
# 2) Check that JAX sees the GPU
import jax
print(jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU! Kaggle: right panel → Session options → Accelerator → GPU T4 x2"


In [ ]:
# 3) Get the project code and the Go2 robot model
import sys, warnings
warnings.simplefilter("ignore")
!rm -rf homeostatic-Alife && git clone -q --depth 1 https://github.com/pablopedrosap/homeostatic-Alife.git
sys.path.insert(0, "homeostatic-Alife/kaggle")
import go2_body_budget as G
xml = G.fetch_go2("mujoco_menagerie")
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT, exist_ok=True)
print("robot:", xml, "| results go to:", OUT)

## Part 0 — speed test
Trains one robot for 3M steps. The first evaluation includes compilation (1–3 min); the speed printed at
the end leaves it out. It tells you how long Part 1 and Part 2 will take on this GPU.

In [ ]:
params, make_inf, info = G.train(xml, homeo_weight=0.2, num_timesteps=3_000_000)
sps = info["steps_per_s"]          # measured after compilation
print(f"\n≈ {sps:,.0f} steps/s  →  50M steps ≈ {50e6 / sps / 60:.0f} min per robot")

## Part 1 — standard vs self-regulating
`weights=(0.0, 0.2)`: drive strength 0 = standard robot, 0.2 = self-regulating. 2 seeds each, 50M steps
per robot, then 3-minute missions with 0, 3 and 6 kg. **Time ≈ 4 × (time per robot from Part 0).**

In [ ]:
rows, models = G.compare_drives(xml, weights=(0.0, 0.2), seeds=(0, 1), num_timesteps=50_000_000,
                                payloads=(0.0, 3.0, 6.0), mission_s=180.0, episodes=3, out_dir=OUT)
table = G.summarize(rows)

In [ ]:
# Plot: distance and motor wear per metre
import numpy as np, matplotlib.pyplot as plt
ws = sorted({r["w"] for r in rows}); kgs = sorted({r["payload"] for r in rows})
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for i, w in enumerate(ws):
    x = np.arange(len(kgs)) + (i - 0.5) * 0.35
    ax[0].bar(x, [np.mean([r["path_m"] for r in rows if r["w"] == w and r["payload"] == k]) for k in kgs], 0.35, label=f"drive w={w:g}")
    ax[1].bar(x, [np.mean([r["wear_per_m"] for r in rows if r["w"] == w and r["payload"] == k]) for k in kgs], 0.35, label=f"drive w={w:g}")
for a, t in zip(ax, ["metres walked in 3 min", "motor wear per metre (lower = better)"]):
    a.set_xticks(range(len(kgs))); a.set_xticklabels([f"{k:g} kg" for k in kgs]); a.set_title(t); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/part1.png", dpi=150); plt.show()

In [ ]:
# Video: the self-regulating robot carrying 3 kg for 20 s
import mediapy
params, make_inf = models["w0.2_seed0"]
m = G.mission(xml, params, make_inf, payload_kg=3.0, seconds=20.0, record_every=2)
frames = G.render(xml, m["frames"])
mediapy.write_video(f"{OUT}/go2_w0.2_3kg.mp4", frames, fps=25)
mediapy.show_video(frames, fps=25)

## Part 2 — evolution of the instinct (long: run it with *Save Version → Save & Run All*)
Two groups of robots. In **evolved**, how much a robot cares about heat (`w`) and its comfort
temperature (setpoint) are inherited and mutate; in **control**, `w = 0`. Every generation each robot
learns to walk from scratch with its inherited instinct (like genes: instincts are inherited, skills
are not), then lives 2 lifetimes of 4 one-minute missions where heat damage permanently weakens its
motors. The best half reproduces.

The cell sizes the run to the hours you give it (Kaggle stops at 12 h in total, and Parts 0–1 use ~4 h
on a T4). Evolution only means something if each robot learns to walk: if the cell warns that there are
too few steps per robot, use fewer robots/generations or a faster GPU.

In [ ]:
HOURS_FOR_PART2 = 6.0            # time budget for this cell
POP, GENERATIONS = 4, 3          # robots per group, generations
MIN_STEPS_TO_WALK = 30_000_000   # update with what Part 1 showed (speed column of the training log)

steps = int(HOURS_FOR_PART2 * 3600 * sps / (2 * POP * GENERATIONS)) // 1_000_000 * 1_000_000
print(f"{sps:,.0f} steps/s → {steps / 1e6:.0f}M steps per robot, {2 * POP * GENERATIONS} robots in {HOURS_FOR_PART2} h")
if steps < MIN_STEPS_TO_WALK:
    print(f"WARNING: fewer than {MIN_STEPS_TO_WALK / 1e6:.0f}M steps per robot: they may not learn to walk. "
          "Lower POP/GENERATIONS or raise HOURS_FOR_PART2.")
history = G.evolve(xml, pop=POP, generations=GENERATIONS, num_timesteps=steps, lifetimes=2, out_dir=OUT)

In [ ]:
# Plot evolution
import numpy as np, matplotlib.pyplot as plt
gens = sorted({h["generation"] for h in history})
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for arm in ("evolved", "control"):
    ax[0].plot(gens, [np.median([h["fitness"] for h in history if h["generation"] == g and h["arm"] == arm]) for g in gens], "o-", label=arm)
ax[1].plot(gens, [np.median([h["w"] for h in history if h["generation"] == g and h["arm"] == "evolved"]) for g in gens], "o-")
ax[0].set_title("lifetime distance (median)"); ax[0].set_xlabel("generation"); ax[0].legend()
ax[1].set_title("evolved instinct strength w (median)"); ax[1].set_xlabel("generation"); ax[1].set_yscale("log")
plt.tight_layout(); plt.savefig(f"{OUT}/part2.png", dpi=150); plt.show()

## Results
Everything is saved in `results/` (Kaggle: **Output** tab → download). Bring `compare_drives.csv`,
`evolution_history.json`, the `.png` plots and the video back to the repository.